# 🦾 Unidad 6 — A2C con Robótica: Brazo Panda

**Curso: Aprendizaje por Refuerzo Profundo en Español**  
Adaptación pedagógica basada en el curso de HuggingFace 🤗  
Autor: Jose Miguel Lara Rangel · [aicraft.mx](https://aicraft.mx)

---

### El primer notebook de robótica real del curso

En este notebook entrenamos un **brazo robótico virtual** para que aprenda a mover
su extremo hasta una posición objetivo en el espacio 3D. Es el primer entorno del
curso con observaciones en forma de **diccionario** y acciones completamente continuas.

| Entorno | Observación | Acciones | Novedad |
|---------|-------------|----------|---------|
| CartPole 🏋️ | 4 números | 2 discretas | — |
| LunarLander 🚀 | 8 números | 4 discretas | — |
| **PandaReach 🦾** | **diccionario (3 claves)** | **3 continuas** | VecNormalize |

Al terminar habrás:
- ✅ Entendido por qué la observación de Panda es un diccionario
- ✅ Aplicado `VecNormalize` — la técnica clave para estabilizar A2C con inputs heterogéneos
- ✅ Entrenado A2C con `MultiInputPolicy` en un entorno de robótica
- ✅ Evaluado correctamente con las estadísticas de normalización cargadas
- ✅ Visualizado el brazo alcanzando su objetivo en video

> 📎 **Slides de referencia:** Capítulos 6.1–6.3 del curso.  
> ⏱️ **Tiempo estimado:** ~30 minutos (la mayor parte es el entrenamiento)

---
## 1 · Configurar el entorno

### 1.1 · Activar la GPU

A2C con Panda converge mucho más rápido con GPU:

**Entorno de ejecución → Cambiar tipo de entorno → GPU T4**

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                         '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode == 0:
    print(f"✅ GPU detectada: {result.stdout.strip()}")
else:
    print("⚠️  Sin GPU — el entrenamiento tardará más. Actívala en el menú de Colab.")

### 1.2 · Instalar dependencias

In [ ]:
%%capture
!apt install -q python-opengl ffmpeg xvfb
!pip install -q pyvirtualdisplay
!pip install -q 'stable-baselines3[extra]'
!pip install -q gymnasium huggingface_sb3 huggingface_hub panda_gym
print("✅ Dependencias instaladas")

> ⚠️ Si Colab solicita reiniciar el entorno después de la instalación, acepta y
> ejecuta desde la sección 1.3.

### 1.3 · Iniciar pantalla virtual

In [ ]:
from pyvirtualdisplay import Display
virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()
print("✅ Pantalla virtual iniciada")

### 1.4 · Importar librerías

In [ ]:
import os
import gymnasium as gym
import panda_gym                                  # los entornos del brazo robótico
import imageio                                    # para grabar video
import numpy as np

from stable_baselines3 import A2C
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor

from huggingface_sb3 import load_from_hub, package_to_hub
from huggingface_hub import notebook_login

print("✅ Librerías importadas")

---
## 2 · Conocer el entorno: PandaReachDense-v3 🦾

### 2.1 · Crear el entorno y explorar su estructura

A diferencia de todos los entornos anteriores donde el estado eran números en un array,
Panda usa un **diccionario** como observación. Veamos por qué:

In [ ]:
env_id = "PandaReachDense-v3"
env = gym.make(env_id)

# Ver el espacio de observaciones
print("═══ ESPACIO DE OBSERVACIONES ═══")
print(env.observation_space)
print()
# Ver una observación real
obs, info = env.reset()
print("Claves del diccionario de observación:", list(obs.keys()))
print()
for key, val in obs.items():
    print(f"  {key}: forma={val.shape}  →  {val}")

### 2.2 · Las 3 claves del diccionario — ¿qué representa cada una?

| Clave | Forma | Descripción | Ejemplo |
|-------|-------|-------------|---------|
| `observation` | (6,) | Posición (x,y,z) y velocidad (vx,vy,vz) del gripper | `[0.3, -0.1, 0.4, 0.0, 0.0, 0.0]` |
| `achieved_goal` | (3,) | Posición actual del gripper en el espacio 3D | `[0.3, -0.1, 0.4]` |
| `desired_goal` | (3,) | Posición del objetivo (donde debe llegar el gripper) | `[0.1, 0.2, 0.3]` |

**Total: 12 valores** (6 + 3 + 3) que la red neuronal debe fusionar para decidir la acción.

### ¿Por qué se divide en 3 grupos en lugar de un solo array de 12 valores?

El formato de diccionario viene de la arquitectura **HER (Hindsight Experience Replay)**,
un algoritmo avanzado de RL para robótica donde separar `achieved_goal` de `desired_goal`
permite reutilizar episodios fallidos como si hubieran sido exitosos.
No usaremos HER en este notebook, pero el formato se mantiene por compatibilidad.

### ¿Por qué `MultiInputPolicy`?

Las políticas `MlpPolicy` (que usamos en CartPole y LunarLander) esperan un tensor plano.
`MultiInputPolicy` maneja automáticamente el diccionario: procesa cada clave con
su propia sub-red y luego fusiona los resultados antes de la capa de salida.

In [ ]:
print("═══ ESPACIO DE ACCIONES ═══")
print(f'Tipo: {env.action_space}')
print(f'Dimensiones: {env.action_space.shape[0]} valores continuos')
print(f'Rango: [{env.action_space.low[0]:.1f}, {env.action_space.high[0]:.1f}]')
print()
print("Cada acción es un vector de 3 números continuos:")
print("  acción[0] = velocidad en eje X  (−1 a +1)")
print("  acción[1] = velocidad en eje Y  (−1 a +1)")
print("  acción[2] = velocidad en eje Z  (−1 a +1)")
print()
print(f"Ejemplo de acción aleatoria: {env.action_space.sample()}")

### 2.3 · El sistema de recompensas

| Tipo | Fórmula | Efecto |
|------|---------|--------|
| **Dense (densa)** | R = −‖gripper − objetivo‖ | El agente recibe señal en **cada paso** proporcional a la distancia. Fácil de aprender. |
| Sparse | R = 0 ó +1 | Solo hay recompensa al llegar. Muy difícil sin RND. |

Usamos `PandaReachDense-v3` (con 'Dense' en el nombre) — igual al ejemplo de las slides.
Un brazo estático obtiene ~−7. Un brazo bien entrenado obtiene ~−1 a −0.5.

---
## 3 · VecNormalize — la pieza más importante de este notebook 🔑

### 3.1 · ¿Por qué normalizar las observaciones?

Las 12 variables del estado de Panda tienen **escalas radicalmente distintas**:

| Variable | Rango típico | Escala |
|----------|-------------|--------|
| Posición gripper (x, y, z) | −1.5 a +1.5 m | ~1.0 |
| Velocidad gripper (vx, vy, vz) | −3.0 a +3.0 m/s | ~1.0 |
| Posición objetivo (x, y, z) | −1.0 a +1.0 m | ~1.0 |

En Panda las escalas son similares, pero en general (ej: Huggy con 243 variables
que incluyen ángulos en radianes, velocidades angulares y posiciones en metros)
las diferencias de escala pueden ser de 100× o más.

`VecNormalize` soluciona esto calculando la **media y desviación estándar** de cada variable
durante el entrenamiento y normalizando automáticamente a media≈0, std≈1.

### 3.2 · Un detalle crítico: guardar y cargar las estadísticas

El problema: si normalizamos las observaciones durante el entrenamiento pero no
guardamos las estadísticas (media y std), al evaluar el modelo el agente recibirá
observaciones en escala diferente a la que aprendió. El resultado: se comportará
de forma completamente errática aunque el modelo en sí esté bien entrenado.

```
Entrenamiento: obs normalizada con media=[0.3, -0.1, ...], std=[0.8, 1.2, ...]
Evaluación sin cargar stats: obs cruda, escala diferente → agente confundido ❌
Evaluación cargando stats: obs normalizada igual → agente funciona bien ✅
```

Por eso guardamos dos archivos: el modelo (`.zip`) **y** las estadísticas (`.pkl`).

### 3.3 · Los parámetros de VecNormalize

| Parámetro | Valor | ¿Qué hace? |
|-----------|-------|-----------|
| `norm_obs` | `True` | Normaliza las observaciones (media≈0, std≈1) |
| `norm_reward` | `True` | Normaliza las recompensas también — estabiliza el crítico |
| `clip_obs` | `10.0` | Recorta observaciones a ±10 std — evita outliers extremos |

---
### 🏋️ EJERCICIO 1 (opcional) — aplicar VecNormalize

Crea el entorno vectorizado con 4 copias y aplica `VecNormalize`.

**Pistas:**
- `make_vec_env(env_id, n_envs=4)` crea 4 entornos en paralelo
- `VecNormalize(env, norm_obs=..., norm_reward=..., clip_obs=...)` es el wrapper
- `norm_obs=True`, `norm_reward=True`, `clip_obs=10.`

Si prefieres no hacerlo como ejercicio, salta a la celda de Solución.

In [ ]:
# 🏋️ EJERCICIO 1 — escribe tu código aquí
# env = make_vec_env(...)
# env = VecNormalize(...)

---
### ✅ Solución

In [ ]:
# ✅ SOLUCIÓN
env = make_vec_env(env_id, n_envs=4)   # 4 brazos robóticos en paralelo

env = VecNormalize(
    env,
    norm_obs=True,     # normalizar observaciones
    norm_reward=True,  # normalizar recompensas (estabiliza el crítico)
    clip_obs=10.       # recortar outliers a ±10 desviaciones estándar
)

print("✅ Entorno con VecNormalize creado")
print(f"Número de entornos en paralelo: {env.num_envs}")
print(f"Normalización de observaciones: {env.norm_obs}")
print(f"Normalización de recompensas: {env.norm_reward}")

---
## 4 · Crear el modelo A2C

### 4.1 · Hiperparámetros de A2C para Panda

Los valores por defecto de SB3 funcionan bien para `PandaReachDense-v3`.
Aquí los más importantes y su justificación:

| Parámetro | Valor por defecto | ¿Qué controla? |
|-----------|------------------|----------------|
| `policy` | `MultiInputPolicy` | Tipo de red — **obligatorio** para observaciones en diccionario |
| `n_steps` | 5 | Pasos antes de cada actualización del gradiente |
| `gamma` | 0.99 | Factor de descuento |
| `learning_rate` | 7e-4 | Tasa de aprendizaje del Actor y el Crítico |
| `ent_coef` | 0.0 | Coeficiente de entropía (0 = sin bonificación de exploración) |
| `vf_coef` | 0.5 | Peso de la pérdida del Crítico en la pérdida total |

---
### 🏋️ EJERCICIO 2 (opcional) — crear el modelo A2C

Crea el modelo con `A2C`. El único parámetro obligatorio que no tiene default es la política.

**Pistas:**
- `A2C(policy=..., env=..., verbose=1)`
- La política correcta para observaciones en diccionario es `'MultiInputPolicy'`

Si prefieres no hacerlo como ejercicio, salta a la celda de Solución.

In [ ]:
# 🏋️ EJERCICIO 2 — escribe tu código aquí
# model = A2C(...)

---
### ✅ Solución

In [ ]:
# ✅ SOLUCIÓN
model = A2C(
    policy='MultiInputPolicy',  # para observaciones en formato diccionario
    env=env,
    verbose=1                   # mostrar progreso en consola
)

print("✅ Modelo A2C creado")
print(f"Política: {model.policy}")
total_params = sum(p.numel() for p in model.policy.parameters())
print(f"Parámetros entrenables: {total_params:,}")

---
## 5 · Entrenar el agente

### ¿Qué esperar durante el entrenamiento?

El log de A2C en SB3 muestra estas métricas clave:

```
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 50       |  → duración media del episodio (pasos)
|    ep_rew_mean        | -5.2     |  → recompensa media — debe subir hacia -1
| time/                 |          |
|    fps                | 1800     |  → pasos por segundo (GPU ≈ 1500-2500)
|    total_timesteps    | 10000    |  → pasos totales completados
| train/                |          |
|    value_loss         | 0.045    |  → pérdida del Crítico — debe bajar
|    policy_gradient_loss | -0.02  |  → pérdida del Actor
------------------------------------
```

| `ep_rew_mean` | Interpretación |
|--------------|----------------|
| −7 a −5 | El brazo apenas se mueve o va en dirección equivocada |
| −4 a −2 | Aprende a acercarse pero no llega consistentemente |
| −2 a −1 | Buena política — llega al objetivo la mayoría de veces |
| **≤ −1.5** | **Meta de certificación HF** |

> ⏱️ Con GPU T4: ~10-15 minutos. Sin GPU: ~45-60 minutos.

In [ ]:
print("Entrenando A2C en PandaReachDense-v3...")
print("Observa cómo ep_rew_mean sube desde ~-5 hasta ~-1 o menos.")
print()
model.learn(total_timesteps=1_000_000)
print()
print("✅ Entrenamiento completado")

### Guardar el modelo Y las estadísticas de normalización

Guardamos **dos archivos separados**. Ambos son necesarios para evaluar y publicar:

In [ ]:
# Guardar el modelo entrenado
model.save("a2c-PandaReachDense-v3")

# Guardar las estadísticas de VecNormalize (media y std de cada variable)
env.save("vec_normalize.pkl")

print("✅ Modelo guardado:       a2c-PandaReachDense-v3.zip")
print("✅ Estadísticas guardadas: vec_normalize.pkl")
print()
print("⚠️  Ambos archivos son necesarios para evaluar correctamente.")
print("   Sin vec_normalize.pkl el agente recibirá observaciones con escala diferente.")

---
## 6 · Evaluar el agente — el patrón con VecNormalize

### El patrón de evaluación con VecNormalize — 4 pasos obligatorios

Evaluar con `VecNormalize` requiere un patrón específico que el original del curso
no explica. Aquí cada paso con su razón:

```python
# PASO 1: Crear el entorno de evaluación base
eval_env = DummyVecEnv([lambda: gym.make(env_id)])
# DummyVecEnv es el wrapper mínimo que SB3 requiere para evaluar.
# Solo 1 entorno (no 4) — la evaluación no necesita paralelización.

# PASO 2: Cargar las MISMAS estadísticas del entrenamiento
eval_env = VecNormalize.load("vec_normalize.pkl", eval_env)
# CRÍTICO: sin esto el agente ve observaciones con escala diferente.

# PASO 3: Desactivar el modo entrenamiento
eval_env.training = False
# En modo entrenamiento VecNormalize actualiza las stats con cada obs.
# En evaluación queremos estadísticas fijas — no queremos que cambien.

# PASO 4: Desactivar normalización de recompensas
eval_env.norm_reward = False
# En evaluación queremos ver la recompensa REAL, no la normalizada.
# Normalizar recompensas durante eval distorsionaría la métrica.
```

In [ ]:
# Construir el entorno de evaluación correctamente
eval_env = DummyVecEnv([lambda: gym.make(env_id)])
eval_env = VecNormalize.load('vec_normalize.pkl', eval_env)
eval_env.training   = False   # no actualizar las estadísticas
eval_env.norm_reward = False  # recompensas reales en evaluación

# Cargar el modelo guardado
model_eval = A2C.load('a2c-PandaReachDense-v3')

# Evaluar en 10 episodios
print("Evaluando el agente (10 episodios)...")
mean_reward, std_reward = evaluate_policy(
    model_eval, eval_env,
    n_eval_episodes=10,
    deterministic=True
)
print(f"\nRecompensa media: {mean_reward:.2f} ± {std_reward:.2f}")

if mean_reward >= -3.5:
    print("✅ ¡Meta de certificación HF alcanzada! (≥ −3.5)")
elif mean_reward >= -5.0:
    print("🟡 Progreso visible. Prueba con más timesteps o ajusta los hiperparámetros.")
else:
    print("🔴 El agente necesita más entrenamiento.")
    print("   Prueba: aumentar n_envs, learning_rate o n_steps.")

---
## 7 · Visualizar el brazo en acción 🎬

Grabamos un episodio completo del brazo usando la política entrenada.

In [ ]:
from IPython.display import Video

def grabar_brazo(env_id, model, stats_path, archivo='replay_panda.mp4', fps=30):
    """Graba un episodio del brazo robótico usando la política entrenada."""
    # Crear entorno para grabación
    env_video = DummyVecEnv([lambda: gym.make(env_id, render_mode='rgb_array')])
    env_video = VecNormalize.load(stats_path, env_video)
    env_video.training   = False
    env_video.norm_reward = False

    frames = []
    obs    = env_video.reset()
    done   = False
    pasos  = 0
    MAX_PASOS = 200

    while not done and pasos < MAX_PASOS:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, info = env_video.step(action)
        frame = env_video.render()
        if frame is not None:
            frames.append(frame)
        pasos += 1

    env_video.close()

    if frames:
        imageio.mimsave(archivo, [np.array(f) for f in frames], fps=fps)
        print(f"Video guardado: {archivo} ({len(frames)} frames, {pasos} pasos)")
        return archivo
    else:
        print("No se generaron frames. Verifica que el entorno soporte render_mode=rgb_array")
        return None

print("Grabando episodio del brazo Panda...")
video = grabar_brazo(env_id, model_eval, 'vec_normalize.pkl')
if video:
    display(Video(video, embed=True, width=500))

---
## 8 · Reto: entrena en otro entorno de Panda 🤖

### Los 4 entornos de Panda disponibles

El mismo código funciona con cualquiera de estos entornos — solo cambia el `env_id`
y posiblemente necesitas más timesteps para los más difíciles:

| Entorno | Tarea | Dificultad | Timesteps sugeridos |
|---------|-------|------------|---------------------|
| `PandaReachDense-v3` ✅ | Mover gripper a posición | ⭐ | 1M |
| `PandaPushDense-v3` | Empujar un cubo a posición | ⭐⭐ | 2M |
| `PandaPickAndPlaceDense-v3` | Agarrar cubo y llevarlo | ⭐⭐⭐ | 3M+ |
| `PandaStackDense-v3` | Apilar dos cubos | ⭐⭐⭐⭐ | 5M+ |

**Reto:** elige `PandaPushDense-v3` o `PandaPickAndPlaceDense-v3` y replica el proceso completo.
El código es casi idéntico — solo cambia `env_id`.

---
### ✅ Solución guiada para PandaPickAndPlace

In [ ]:
# Reto: PandaPickAndPlace — agarrar y colocar un cubo
# Descomenta para ejecutar (requiere ~30-60 min con GPU)

# env_id_reto = 'PandaPickAndPlaceDense-v3'
# env_reto = make_vec_env(env_id_reto, n_envs=4)
# env_reto = VecNormalize(env_reto, norm_obs=True, norm_reward=True, clip_obs=10.)

# model_reto = A2C(policy='MultiInputPolicy', env=env_reto, verbose=1)
# model_reto.learn(total_timesteps=3_000_000)  # necesita más pasos

# model_reto.save('a2c-PandaPickAndPlaceDense-v3')
# env_reto.save('vec_normalize_pickplace.pkl')

# eval_env_reto = DummyVecEnv([lambda: gym.make(env_id_reto)])
# eval_env_reto = VecNormalize.load('vec_normalize_pickplace.pkl', eval_env_reto)
# eval_env_reto.training    = False
# eval_env_reto.norm_reward = False

# mean_reto, std_reto = evaluate_policy(model_reto, eval_env_reto, n_eval_episodes=10)
# print(f"PandaPickAndPlace: {mean_reto:.2f} ± {std_reto:.2f}")
print("Descomenta las líneas de arriba para ejecutar el reto.")

---

# 🔒 Sección opcional — Publicar en el Hub

> Requiere cuenta de HuggingFace.  
> La meta de certificación es recompensa media ≥ −3.5.

---

## [OPCIONAL] Publicar tu brazo en el Hugging Face Hub 🤗

### Paso 1: Iniciar sesión

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # pega tu token con permiso Write
!git config --global credential.helper store

### Paso 2: Publicar el modelo

Con Panda, `package_to_hub` necesita el entorno de evaluación con VecNormalize ya configurado
(el mismo que usamos para evaluar). Reemplaza `TU_USUARIO` con tu nombre de HuggingFace:

In [ ]:
USERNAME = ''   # ← pon tu usuario aquí

# Asegúrate de que eval_env está configurado (sección 6)
package_to_hub(
    model=model_eval,
    model_name=f"a2c-{env_id}",
    model_architecture="A2C",
    env_id=env_id,
    eval_env=eval_env,
    repo_id=f"{USERNAME}/a2c-{env_id}",
    commit_message="Brazo robótico Panda entrenado con A2C 🦾"
)

---

## 🎉 ¡Notebook completado!

Entrenaste un brazo robótico real con A2C. Lo más importante que aprendiste:

1. **Observaciones en diccionario** — por qué Panda las necesita y cómo `MultiInputPolicy` las maneja
2. **`VecNormalize`** — normalización automática de inputs heterogéneos durante el entrenamiento
3. **El patrón de evaluación con stats** — por qué guardar y cargar `vec_normalize.pkl` es crítico
4. **Dense vs. Sparse** — cómo el diseño de la función de recompensa facilita o dificulta el aprendizaje
5. **A2C en robótica** — el mismo algoritmo que estudiamos teóricamente, aplicado a un brazo 3D

### ¿Qué sigue?

- 📓 **Notebook 8:** PPO desde cero con CleanRL — el algoritmo que mejora A2C con el clip
- 📊 **Slides M6:** ¿Por qué A2C puede dar pasos demasiado grandes? → la motivación de PPO

---
*Aicraft · aicraft.mx · Jose Miguel Lara Rangel*